**Imports**

This notebook depends on some open source software libraries, and some helper functions in `draw_map.py`

- PolaRS: Efficient processing of big tables (dataframes)
- KML: Map markup language
- ETree and BeautifulSoup: read the XML and HTML formats within the KML data
- ZipFS: Compressed filesystem reader
- Leaflet: Map drawing

In [1]:
import polars as pl
from fastkml import kml
from fs.zipfs import ZipFS
from lxml import etree
from bs4 import BeautifulSoup
from ipyleaflet import (Polyline, Map,basemaps)

**Seattle Street Map Data**

Go to Seattle City GIS: SDOT Bike Facilities:

https://data-seattlecitygis.opendata.arcgis.com/maps/SeattleCityGIS::sdot-bike-facilities/about

Choose which map you want and click through to the ArcGIS viewer:
- Multi-use Trails
- Existing Bike Facilities
- Planned Bike Facilities

Click "Download" and select KML"
Change the path below to point to your file.

In [5]:
coords = (47.6131746,-122.4878834) # Seattle
kmz_file_path = '/path/to/kmz/data/SDOT_Bike_Facilities_000000000000.kmz' # Change this to the file you downloaded

**Read the compressed KML file**

In [6]:
with ZipFS(kmz_file_path) as home_fs:
    with home_fs.open('doc.kml') as f:
        root = etree.parse(f)
        kml_parser = kml.KML()
        kml_parser.from_string(
            etree.tostring(root)
        )

**Define helper functions for reading the description and line information from the KML format**

In [7]:
def description_dict(description_html):
    soup = BeautifulSoup(description_html, 'html.parser')
    d_table =  soup.body.table

    d_dict = {}

    for tr in d_table.find_all('tr'):
        try:
            (k, v) = tr.find_all('td')
            d_dict[k.text] = v.text
        except:
            continue

    return d_dict

def full_dict(feature):
    data = description_dict(feature.description)

    data['name'] = feature.name
    data['polyline'] = [(g.y, g.x) for g in feature.geometry.geoms]

    return data

def get_description_data(kml_parser):
    level = next(
        next(
            kml_parser.features()
        )
        .features()
    )

    for f in level.features():
        yield full_dict(f)

**Convert the data to a dataframe using the helper functions**

In [8]:
description_df = pl.DataFrame(get_description_data(kml_parser))

**Filter a subset of the segments**

In [9]:
greenways_df = (
    description_df
    .select(
        pl.col('polyline'),
        pl.when(
            pl.col('CATEGORY').is_in(['BKF-PBL', 'BKF-BL', 'BKF-SHW'])
        )
        .then(
            pl.lit('e')
        )
        .otherwise(
            pl.when(
                pl.col('CATEGORY').is_in(['BKF-NGW'])
            )
            .then(
                pl.lit('i')
            )
            .otherwise(
                pl.when(
                    pl.col('CATEGORY').is_in([])
                )
                .then(
                    pl.lit('d')
                )
                .otherwise(
                    pl.lit('u')
                )
            )
        )
        .alias('difficulty')
    )
)

**Define helper functions for drawing the map**

In [10]:
UNCLASSIFIED = '#808080'
ALL_ABILITIES = '#00FF00'
INTERMEDIATE = '#FFFF00'
DIFFICULT = '#FF0000'

difficulties = { 'u' : UNCLASSIFIED,
                 'e' : ALL_ABILITIES,
                 'i' : INTERMEDIATE,
                 'd' : DIFFICULT }

def draw_lines(folium_map, segment_df):
    LINE_WEIGHT=3
    OPACITY=1.0

    for (poly_coords, difficulty) in segment_df.iter_rows():
        line_color = difficulties[difficulty]
        l = Polyline(
            locations=poly_coords,
            color=line_color,
            fill_color=line_color,
            weight=LINE_WEIGHT,
            opacity=OPACITY)
        folium_map += l

def get_map(coords):
    _BACKGROUND=basemaps.Esri.NatGeoWorldMap

    get_map.folium_map = Map(
        center=coords, zoom=10,
        basemap=_BACKGROUND)

    return get_map.folium_map

**Draw the map, save it as an HTML file and a PNG and show it in the notebook**

In [14]:
m = get_map(coords)
draw_lines(m, greenways_df)
m.save('example_2025.html')
m

Map(center=[47.6131746, -122.4878834], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_tit…

**Workaround: run chrome headless (Mac version of command) to export the image**

In [21]:
!"/Applications/Google Chrome.app/Contents/MacOS/Google Chrome" --headless --start-maximized --start-fullscreen --window-size="2000,1000" --hide-scrollbars --virtual-time-budget=5000 --screenshot=example_2025.png  example_2025.html && sleep 6

[24168:259:0404/171923.140360:ERROR:chrome_browser_main.cc(1143)] The use of Rosetta to run the x64 version of Chromium on Arm is neither tested nor maintained, and unexpected behavior will likely result. Please check that all tools that spawn Chromium are Arm-native.
[24168:47107:0404/171926.262406:ERROR:trust_store_mac.cc(817)] Error parsing certificate:
ERROR: Failed parsing extensions

Created TensorFlow Lite XNNPACK delegate for CPU.
Attempting to use a delegate that only supports static-sized tensors with a graph that has dynamic-sized tensors (tensor#-1 is a dynamic-sized tensor).
3440979 bytes written to file example_2025.png


![png](example_2025.png)